In [4]:
import awkward as ak
import uproot
import uproot3
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema
from coffea.nanoevents.methods import vector

In [5]:
# fileset = {
#     'WJetsToLNu_Pt-400To600_MatchEWPDG20_TuneCP5_13TeV-amcatnloFXFX-pythia8____1_':
# [
# "root://cmseos.fnal.gov//store/group/lpcmetx/Monotop/CustomNanoProdV9/2016/Mphi-1995_Mchi-1000_2016/Mphi-1995_Mchi-1000_2016_1.root",
# "root://cmseos.fnal.gov//store/group/lpcmetx/Monotop/CustomNanoProdV9/2016APV/Mphi-1995_Mchi-1000_2016/Mphi-1995_Mchi-1000_2016_1.root"
# ]}

file_2016_UL = [
            "root://xrootd-cms.infn.it//store/data/Run2016B/SingleMuon/NANOAOD/ver2_HIPM_UL2016_MiniAODv2_NanoAODv9-v1/70000/55BB0652-F28A-8C44-8EF5-C8DC05A1FCE8.root",
            "root://xrootd-cms.infn.it//store/data/Run2016B/SingleMuon/NANOAOD/ver2_HIPM_UL2016_MiniAODv2_NanoAODv9-v1/70000/9713F817-B8FF-C048-94FF-9F45A5750A1E.root",
            "root://cmsxrootd.fnal.gov//store/data/Run2016B/SingleMuon/NANOAOD/ver2_HIPM_UL2016_MiniAODv2_NanoAODv9-v1/70000/4A97CBF1-A25D-294D-99DB-D874853F9C64.root",
    "root://cmsxrootd.fnal.gov//store/data/Run2016B/SingleElectron/NANOAOD/ver2_HIPM_UL2016_MiniAODv2_NanoAODv9-v1/70000/2C477666-3961-EE40-9ABB-9413C2BA6219.root",
    "root://xrootd-cms.infn.it//store/data/Run2016B/SingleMuon/NANOAOD/ver2_HIPM_UL2016_MiniAODv2_NanoAODv9-v1/130000/B15FFF7B-30C2-2449-915B-A172881A22B2.root"
        ]
# fname = fileset["WJetsToLNu_Pt-400To600_MatchEWPDG20_TuneCP5_13TeV-amcatnloFXFX-pythia8____1_"][1]
fname_ul = file_2016_UL[4]
events = NanoEventsFactory.from_root(fname_ul, schemaclass=NanoAODSchema).events()

In [46]:

from coffea import util
# Initialize physics objects
###
import awkward
import numpy as np
year = '2016'
ids = util.load('../../data/ids.coffea')
isLooseMuon = ids['isLooseMuon']
isTightMuon = ids['isTightMuon']
isLooseElectron = ids['isLooseElectron']
isTightElectron = ids['isTightElectron']
isLoosePhoton = ids['isLoosePhoton']
isTightPhoton = ids['isTightPhoton']
isGoodJet = ids['isGoodJet']
isHEMJet = ids['isHEMJet']
mu = events.Muon
mu['isloose'] = isLooseMuon(mu.pt, mu.eta, mu.pfRelIso04_all, mu.looseId, year)
mu['istight'] = isTightMuon(mu.pt, mu.eta, mu.pfRelIso04_all, mu.tightId, year)
mu["T"] = ak.zip({"pt": mu.pt, "phi": mu.phi}, 
          with_name="PolarTwoVector", 
          behavior=vector.behavior)
mu['p4'] = ak.zip({
                    "pt": mu.pt,
                    "eta": mu.eta,
                    "phi": mu.phi,
                    "mass": mu.mass},
                    with_name="PtEtaPhiMLorentzVector",
)


mu_loose = mu[ak.values_astype(mu.isloose, np.bool)]
mu_tight = mu[ak.values_astype(mu.istight, np.bool)]
mu_ntot = ak.num(mu, axis=1)
mu_nloose = ak.num(mu_loose, axis=1)
mu_ntight = ak.num(mu_tight, axis=1)
leading_mu = mu[:,:1]
leading_mu = leading_mu[ak.values_astype(leading_mu.istight, np.bool)]


e = events.Electron
event_size = len(events)
e['isclean'] = ~(ak.any(e.metric_table(mu_loose) < 0.3, axis=-1))
e['isloose'] = isLooseElectron(e.pt, e.eta+e.deltaEtaSC, e.dxy, e.dz, e.cutBased, year)
e['istight'] = isTightElectron(e.pt, e.eta+e.deltaEtaSC, e.dxy, e.dz, e.cutBased, year)
#         e['T'] = TVector2Array.from_polar(e.pt, e.phi)
e["T"] = ak.zip({"pt": e.pt, "phi": e.phi}, 
          with_name="PolarTwoVector", 
          behavior=vector.behavior)
e['p4'] = ak.zip({
                    "pt": e.pt,
                    "eta": e.eta,
                    "phi": e.phi,
                    "mass": e.mass},
                    with_name="PtEtaPhiMLorentzVector",
)
e_clean = e[ak.values_astype(e.isclean, np.bool)]
e_loose = e_clean[ak.values_astype(e_clean.isloose, np.bool)]
e_tight = e_clean[ak.values_astype(e_clean.istight, np.bool)]
e_ntot = ak.num(e, axis=1)
e_nloose = ak.num(e_loose, axis=1)

#         e_nloose = e_loose.counts
e_ntight = ak.num(e_tight, axis=1)
leading_e = e[:,:1]
leading_e = leading_e[ak.values_astype(leading_e.isclean, np.bool)]
leading_e = leading_e[ak.values_astype(leading_e.istight, np.bool)]




pho = events.Photon
pho['isclean'] = ~(ak.any(pho.metric_table(mu_loose) < 0.4, axis=-1)) & ~(ak.any(pho.metric_table(e_loose) < 0.4, axis=-1))
_id = 'cutBased'
if year == '2016':
    _id = 'cutBased'
pho['isloose'] = isLoosePhoton(pho.pt, pho.eta, pho[_id], year) & (pho.electronVeto)  # added electron veto flag
pho['istight'] = isTightPhoton(pho.pt, pho[_id], year) & (pho.isScEtaEB) & (pho.electronVeto)  # tight photons are barrel only

pho["T"] = ak.zip({"pt": pho.pt, "phi": pho.phi}, 
          with_name="PolarTwoVector", 
          behavior=vector.behavior)

pho_clean = pho[ak.values_astype(pho.isclean, np.bool)]
pho_loose = pho_clean[ak.values_astype(pho_clean.isloose, np.bool)]
pho_tight = pho_clean[ak.values_astype(pho_clean.istight, np.bool)]
pho_ntot = ak.num(pho,axis=1)
pho_nloose = ak.num(pho_loose, axis=1)
pho_ntight = ak.num(pho_tight, axis=1)
leading_pho = pho[:,:1] #new way to define leading photon
leading_pho = leading_pho[ak.values_astype(leading_pho.isclean, np.bool)]
leading_pho = leading_pho[ak.values_astype(leading_pho.istight, np.bool)]

j = events.Jet
def isGoodJet_(pt, eta, jet_id, pu_id,NHF, NEMF, NumConst, MUF, CHF, CHM, CEMF, year):
    #working id for tt region
#     if year=='2016':
#         mask = (pt>=30) & (abs(eta)<=2.4) & (NHF<0.90) & (NEMF<0.90) & (NumConst>1) & (MUF<0.8) & (CHF>0) & (CHM>0) & (CEMF<0.8) #jetId==6 means: pass tight and tightLepVeto ID. https://twiki.cern.ch/twiki/bin/viewauth/CMS/JetID
#         mask = ((pt>=50)&mask) | ((pt<50)&mask&((pu_id&1)==1)) # puId&4)==4 jet pass atleast the loose pu_id, https://twiki.cern.ch/twiki/bin/viewauth/CMS/PileupJetID#Recommendations_for_13_TeV_Ultra

#     else:
#         mask = (pt>30) & (abs(eta)<2.4) & ((jet_id&2)==2) #& (nhf<0.9) & (chf>0.0) #& (nef<0.99) & (cef<0.99)
#         mask = ((pt>=50)&mask) | ((pt<50)&mask&((pu_id&1)==1))
# ((jet_id&2)==2) -->> equivalent to ((jet_id>=2) tight and tighlepveto

    if year=='2016':
        mask = (pt>=30) & ((jet_id&2)==2)& (abs(eta)<=2.4) & (NHF<0.90) & (NEMF<0.90) & (NumConst>1)  & (CHF>0) & (CHM>0)
        mask = ((pt>=50)&mask) | ((pt<50)&mask&((pu_id==7)))
#     else:
#         mask = (pt>=30) & (abs(eta)<=2.6) & ((jet_id&2)==2)& (NHF<0.90) & (NEMF<0.90) & (NumConst>1) & (MUF<0.8) & (CHF>0) & (CHM>0) & (CEMF<0.8)
#         mask = ((pt>=50)&mask) | ((pt<50)&mask&((pu_id&1)==1))
#     return mask
#     if year=='2016':
#         mask = (pt>=30) & (jet_id>=4) & (abs(eta)<=2.4)
#         mask = ((pt>=50)&mask) | ((pt<50)&mask&((pu_id&1)==1))
    else:
        mask = (pt>=30) & (abs(eta)<=2.6) & ((jet_id&2)==2)#& (NHF<0.90) & (NEMF<0.90) & (NumConst>1) & (MUF<0.8) & (CHF>0) & (CHM>0) & (CEMF<0.8)
        mask = ((pt>=50)&mask) | ((pt<50)&mask&((pu_id&1)==1))
    return mask
def jetcleaning(j, mu_loose, e_loose, pho_loose, r):
    NumNeutralParticles = (j.nConstituents-j.nElectrons-j.nMuons)
    NumConst = j.nConstituents
    NHF = j.neHEF
    NEMF = j.neEmEF
    CHF = j.chHEF
    MUF = j.muEF
    CEMF = j.chEmEF
    CHM = j.nElectrons+j.nMuons

    jetMuMask = ak.any(j.metric_table(mu_loose) < r, axis=-1)
    jetEleMask = ak.any(j.metric_table(e_loose) < r, axis=-1)
    jetPhoMask = ak.any(j.metric_table(pho_loose) < r, axis=-1)

    j_isclean_mask = (~jetMuMask & ~jetEleMask & ~jetPhoMask)
    j_isgood_mask = isGoodJet_(j.pt, j.eta, j.jetId, j.puId,NHF, NEMF, NumConst, MUF, CHF, CHM, CEMF, year=year)
    j_good_clean = j[j_isclean_mask & j_isgood_mask]

    return(j_good_clean)

#         j['isgood'] = isGoodJet(j.pt, j.eta, j.jetId, j.puId,NHF, NEMF, NumConst, MUF, CHF, CHM, CEMF, year=year)
j['isHEM'] = isHEMJet(j.pt, j.eta, j.phi)
# j['isdcsvL'] = (j.btagDeepB>deepcsvWPs['loose'])
# j['isdflvL'] = (j.btagDeepFlavB>deepflavWPs['loose'])
# j['isdflvM'] = (j.btagDeepFlavB > deepflavWPs['medium']) ## from Rishab
# j['isdcsvM'] = (j.btagDeepB > deepcsvWPs['medium']) ## from Rishabh

j["T"] = ak.zip({"pt": j.pt, "phi": j.phi}, 
          with_name="PolarTwoVector", 
          behavior=vector.behavior)
j['p4'] = ak.zip({
"pt": j.pt,
"eta": j.eta,
"phi": j.phi,
"mass": j.mass},
with_name="PtEtaPhiMLorentzVector",
)

#         https://coffeateam.github.io/coffea/modules/coffea.nanoevents.methods.vector.html#
#         j['ptRaw'] =j.pt * (1-j.rawFactor)
#         j['massRaw'] = j.mass * (1-j.rawFactor)
#         j['rho'] = j.pt.ones_like()*events.fixedGridRhoFastjetAll.array

#         j_dflvL = j_clean[j_clean.isdflvL.astype(np.bool)]


j_good_clean = jetcleaning(j, mu_loose, e_loose, pho_loose, 0.4)
j_ngood_clean = ak.num(j_good_clean)
# j_good_clean_dflvB = j_good_clean.isdflvM
# j_ndflvM = ak.num(j[j_good_clean_dflvB], axis=1)
# leading_j = j_good_clean[:,:1] # new way to define leading jet
# j_HEM = j[ak.values_astype(j.isHEM, np.bool)]       
# j_nHEM = ak.num(j_HEM, axis=1)
atleast_one_jet_with_pt_grt_50 = ((ak.num(j_good_clean)>=1) & ak.any(j_good_clean.pt>70, axis=-1))

In [49]:
ak.sum(atleast_one_jet_with_pt_grt_50 & (j_ndflvM==0))

208331

In [37]:
j_ndflvM = ak.num(j[j_good_clean['btagDeepB'] > btagWP_medium], axis=1)

In [38]:
j_good_clean['btagDeepB']

<Array [[0.7], [], [], ... [], [0.927, 0.61]] type='2755162 * var * float32[para...'>

In [39]:
btagWP_medium = 0.4184
leading_j_b = j_good_clean[~(j_good_clean['btagDeepB'] > btagWP_medium)][:,:1]

In [40]:
leading_j_b

<JetArray [[], [], [], [], ... [], [], [], []] type='2755162 * var * jet'>

In [45]:
ak.sum(ak.flatten((leading_j_b.pt>70) & (j_ndflvM==0)))

208331

In [42]:
leading_j_not_b = j_good_clean[~(j_good_clean['btagDeepB'] > btagWP_medium)][:,:1]

In [43]:
ak.sum(ak.flatten((leading_j_not_b.pt>70)& (j_ndflvM==0)))

208331

In [44]:
ak.flatten(j_good_clean[(leading_j_not_b.pt>70)& (j_ndflvM==0)].pt)

<Array [88.4, 107, 71.4, ... 89.1, 176, 78.8] type='208331 * float32[parameters=...'>

In [47]:
# (pt>=30) & (abs(eta)<=2.4) & ((jet_id&2)==2)& (NHF<0.90) & (NEMF<0.90) & (NumConst>1) & (MUF<0.8) & (CHF>0) & (CHM>0) & (CEMF<0.8)
sum(j_ngood_clean)

2200115

In [48]:
# (pt>=30) & (abs(eta)<=2.4) & (NHF<0.90) & (NEMF<0.90) & (NumConst>1) & (MUF<0.8) & (CHF>0) & (CHM>0) & (CEMF<0.8)
sum(j_ngood_clean)

2200115

In [21]:
j.jetId

<Array [[6, 6, 6], [6, ... 6, 6, 6, 6, 6, 6]] type='2755162 * var * int32[parame...'>

In [32]:
j.puId&1<<2

<Array [[4, 4, 4], [4, ... 4, 4, 4, 0, 4, 4]] type='2755162 * var * int32'>

In [28]:
j.puId


<Array [[7, 7, 7], [7, ... 7, 7, 7, 1, 7, 7]] type='2755162 * var * int32[parame...'>

In [29]:
1<<2

4

In [38]:
set(ak.flatten(j.jetId))

{0, 2, 6}

In [39]:
set(ak.flatten(j.puId))

{0, 1, 3, 7}

In [36]:
j.jetId

<Array [[6, 6, 6], [6, ... 6, 6, 6, 6, 6, 6]] type='2755162 * var * int32[parame...'>

In [33]:
mediumJetIDbit = 1
(j.jetId >> mediumJetIDbit & 1) == 1

<Array [[True, True, True, ... True, True]] type='2755162 * var * bool'>

In [35]:
j.jetId >> mediumJetIDbit

<Array [[3, 3, 3], [3, ... 3, 3, 3, 3, 3, 3]] type='2755162 * var * int32'>